In [18]:
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications import EfficientNetB0

from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint,ReduceLROnPlateau
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

##### Adjust BASE_DIR to your project root

In [19]:
import os
import shutil
from pathlib import Path

# Path to the Keras directory
keras_dir = Path.home() / '.keras'
# Path to the models cache
models_dir = keras_dir / 'models'

# Check if the directory exists and remove it
if models_dir.exists():
    print(f"Found Keras models cache at: {models_dir}")
    shutil.rmtree(models_dir)
    print("Successfully cleared Keras models cache.")
else:
    print("Keras models cache directory not found (already clean).")

Found Keras models cache at: C:\Users\Varun\.keras\models
Successfully cleared Keras models cache.


In [20]:
BASE_DIR = r"C:\Users\Varun\OneDrive\Pictures\Desktop\waste_classification_project\application"
SPLIT_DIR = os.path.join(BASE_DIR, "data", "split")

#### 1 Data Generators

Assign the train,val,test data paths

In [21]:
train_dir = os.path.join(SPLIT_DIR, "train")
val_dir   = os.path.join(SPLIT_DIR, "val")
test_dir  = os.path.join(SPLIT_DIR, "test")

In [22]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
EPOCHS = 10

In [23]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    fill_mode='nearest'
)

In [24]:
val_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

In [25]:
if not os.path.exists(train_dir):
    print(f"Train directory not found: {train_dir}")
else:
    train_generator = train_datagen.flow_from_directory(
        train_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        color_mode='rgb'
        
    )

Found 16271 images belonging to 10 classes.


In [26]:
if not os.path.exists(val_dir):
    print(f"Validation directory not found: {val_dir}")
else:
    val_generator = val_datagen.flow_from_directory(
        val_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        color_mode='rgb'
        )


Found 3483 images belonging to 10 classes.


In [27]:
if not os.path.exists(test_dir):
    print(f"Test directory not found: {test_dir}")
else:
    test_generator = val_datagen.flow_from_directory(
        test_dir,
        target_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        color_mode='rgb'
        )

Found 3498 images belonging to 10 classes.


In [28]:
num_classes = len(train_generator.class_indices)
num_classes

10

In [29]:
print("Class indices:", train_generator.class_indices)

Class indices: {'battery': 0, 'cardboard waste': 1, 'clothe waste': 2, 'glass waste': 3, 'metal waste': 4, 'organic waste': 5, 'paper waste': 6, 'plastic waste': 7, 'shoes waste': 8, 'trash': 9}


#### 2️ Build Transfer Learning Model (EfficientNetB0 )

In [30]:
# Check the shape of images from train_generator
batch_x, batch_y = next(train_generator)
print("Batch image shape:", batch_x.shape)  # Should be (batch_size, 224, 224, 3)

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224,224,3))
base_model.trainable = False  # freeze base layers

Batch image shape: (32, 224, 224, 3)
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 5s 1us/step


In [31]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.4)(x)
predictions = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)

model.compile(optimizer=Adam(learning_rate=0.001),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

#### 3️ Callbacks

In [32]:

os.makedirs(os.path.join(BASE_DIR, "models"), exist_ok=True)
checkpoint_path = os.path.join(BASE_DIR, "models", "waste_model.keras")

callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_loss'),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2)
]


#### 4️ First train only the top layer

In [33]:

history = model.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks
)


Epoch 1/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 892s 2s/step - accuracy: 0.7143 - loss: 0.8493 - val_accuracy: 0.8691 - val_loss: 0.4085 - learning_rate: 0.0010
Epoch 2/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 875s 2s/step - accuracy: 0.8193 - loss: 0.5462 - val_accuracy: 0.8923 - val_loss: 0.3429 - learning_rate: 0.0010
Epoch 3/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 875s 2s/step - accuracy: 0.8280 - loss: 0.5019 - val_accuracy: 0.8964 - val_loss: 0.3317 - learning_rate: 0.0010
Epoch 4/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 874s 2s/step - accuracy: 0.8355 - loss: 0.4921 - val_accuracy: 0.8989 - val_loss: 0.3219 - learning_rate: 0.0010
Epoch 5/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 868s 2s/step - accuracy: 0.8407 - loss: 0.4803 - val_accuracy: 0.8995 - val_loss: 0.3243 - learning_rate: 0.0010
Epoch 6/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 907s 2s/step - accuracy: 0.8405 - loss: 0.4841 - val_accuracy: 0.8989 - val_loss: 0.3234 - learning_rate: 0.0010
Epoch 7/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 869s 2s/step - accuracy: 0.8490 - loss: 0.

In [34]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy: {test_acc*100:.2f}%")

110/110 ━━━━━━━━━━━━━━━━━━━━ 133s 1s/step - accuracy: 0.9125 - loss: 0.2713
Test Accuracy: 91.25%


#### 5️ Then Fine-tune: unfreeze last 50 layers and train with low LR

In [35]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False
    
model.compile(optimizer=Adam(learning_rate=0.0001), loss='categorical_crossentropy', metrics=['accuracy'])

In [36]:
history_fine = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=callbacks
)


Epoch 1/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1107s 2s/step - accuracy: 0.8318 - loss: 0.5116 - val_accuracy: 0.8889 - val_loss: 0.3937 - learning_rate: 1.0000e-04
Epoch 2/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1112s 2s/step - accuracy: 0.8912 - loss: 0.3213 - val_accuracy: 0.9165 - val_loss: 0.3057 - learning_rate: 1.0000e-04
Epoch 3/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1089s 2s/step - accuracy: 0.9247 - loss: 0.2230 - val_accuracy: 0.9360 - val_loss: 0.2212 - learning_rate: 5.0000e-05
Epoch 4/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1086s 2s/step - accuracy: 0.9375 - loss: 0.1867 - val_accuracy: 0.9440 - val_loss: 0.1876 - learning_rate: 5.0000e-05
Epoch 5/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1093s 2s/step - accuracy: 0.9437 - loss: 0.1668 - val_accuracy: 0.9515 - val_loss: 0.1672 - learning_rate: 5.0000e-05
Epoch 6/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1120s 2s/step - accuracy: 0.9494 - loss: 0.1494 - val_accuracy: 0.9480 - val_loss: 0.1813 - learning_rate: 5.0000e-05
Epoch 7/10
509/509 ━━━━━━━━━━━━━━━━━━━━ 1095s 2s/ste

#### 6 Evaluate on test set

In [37]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test Accuracy after fine-tuning: {test_acc*100:.2f}%")

110/110 ━━━━━━━━━━━━━━━━━━━━ 131s 1s/step - accuracy: 0.9651 - loss: 0.1169
Test Accuracy after fine-tuning: 96.51%


Before applying data augmentation, the model achieved a test accuracy of around 75%. After applying data augmentation techniques such as rotation, width/height shifts, shear, zoom, horizontal flips, and brightness adjustments, the model's performance improved significantly. The test accuracy increased to approximately 85%, demonstrating the effectiveness of data augmentation in enhancing the model's ability to generalize to unseen data.

In [39]:
model_path = os.path.join(BASE_DIR, "models", "final_waste_model.keras")
model.save(model_path)
print(f"Model saved successfully ✅ at {model_path}")

Model saved successfully ✅ at C:\Users\Varun\OneDrive\Pictures\Desktop\waste_classification_project\application\models\final_waste_model.keras
